# Real-ESRGAN 4x Batch Upscaler (Pure PyTorch - Zero Dependency Issues)

Upscale kumpulan gambar secara massal dengan GPU T4 di Google Colab.  
- **Model**: RealESRGAN_x4plus (Anti-blur, pertajam detail & tekstur).
- **Format Output**: JPG Kualitas Tinggi (Quality 96, Subsampling 4:4:4, ukuran file padat dan tidak drop).
- **Tiling**: 512x512 (Aman dari CUDA Out-of-Memory untuk gambar resolusi tinggi).
- **Engine**: Pure PyTorch (Tanpa instalasi library luar, anti error basicsr).

---  
**Pastikan Runtime GPU Aktif**:  
Klik menu `Runtime` -> `Change runtime type` -> pilih `T4 GPU` -> `Save`.

In [ ]:
# 1. Periksa GPU (Harus muncul NVIDIA Tesla T4)
!nvidia-smi

In [ ]:
# 2. Siapkan folder dan unduh bobot model RealESRGAN_x4plus
import os
!mkdir -p weights inputs outputs

print("Mengunduh model RealESRGAN_x4plus.pth...")
!wget -c -q --show-progress "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth" -P weights/
print("Model siap digunakan!")

In [ ]:
# 3. Buat script runner mandiri (Pure PyTorch, tanpa basicsr)
%%writefile upscale_batch.py
import argparse
import glob
import math
import os
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

class ResidualDenseBlock(nn.Module):
    def __init__(self, num_feat=64, num_grow_ch=32):
        super().__init__()
        self.conv1 = nn.Conv2d(num_feat, num_grow_ch, 3, 1, 1)
        self.conv2 = nn.Conv2d(num_feat + num_grow_ch, num_grow_ch, 3, 1, 1)
        self.conv3 = nn.Conv2d(num_feat + 2 * num_grow_ch, num_grow_ch, 3, 1, 1)
        self.conv4 = nn.Conv2d(num_feat + 3 * num_grow_ch, num_grow_ch, 3, 1, 1)
        self.conv5 = nn.Conv2d(num_feat + 4 * num_grow_ch, num_feat, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)

    def forward(self, x):
        x1 = self.lrelu(self.conv1(x))
        x2 = self.lrelu(self.conv2(torch.cat((x, x1), 1)))
        x3 = self.lrelu(self.conv3(torch.cat((x, x1, x2), 1)))
        x4 = self.lrelu(self.conv4(torch.cat((x, x1, x2, x3), 1)))
        x5 = self.conv5(torch.cat((x, x1, x2, x3, x4), 1))
        return x5 * 0.2 + x

class RRDB(nn.Module):
    def __init__(self, num_feat=64, num_grow_ch=32):
        super().__init__()
        self.rdb1 = ResidualDenseBlock(num_feat, num_grow_ch)
        self.rdb2 = ResidualDenseBlock(num_feat, num_grow_ch)
        self.rdb3 = ResidualDenseBlock(num_feat, num_grow_ch)

    def forward(self, x):
        out = self.rdb1(x)
        out = self.rdb2(out)
        out = self.rdb3(out)
        return out * 0.2 + x

class RRDBNet(nn.Module):
    def __init__(self, num_in_ch=3, num_out_ch=3, scale=4, num_feat=64, num_block=23, num_grow_ch=32):
        super().__init__()
        self.scale = scale
        self.conv_first = nn.Conv2d(num_in_ch, num_feat, 3, 1, 1)
        self.body = nn.Sequential(*[RRDB(num_feat, num_grow_ch) for _ in range(num_block)])
        self.conv_body = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_up1 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_up2 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_hr = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_last = nn.Conv2d(num_feat, num_out_ch, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)

    def forward(self, x):
        feat = self.conv_first(x)
        body_feat = self.conv_body(self.body(feat))
        feat = feat + body_feat
        feat = self.lrelu(self.conv_up1(F.interpolate(feat, scale_factor=2, mode="nearest")))
        feat = self.lrelu(self.conv_up2(F.interpolate(feat, scale_factor=2, mode="nearest")))
        out = self.conv_last(self.lrelu(self.conv_hr(feat)))
        return out

def load_model(model_path, device, fp16=True):
    model = RRDBNet(num_in_ch=3, num_out_ch=3, scale=4, num_feat=64, num_block=23, num_grow_ch=32)
    checkpoint = torch.load(model_path, map_location=device, weights_only=True)
    state_dict = checkpoint.get("params_ema", checkpoint.get("params", checkpoint))
    model.load_state_dict(state_dict, strict=True)
    model.eval().to(device)
    return model.half() if (fp16 and device.type == "cuda") else model

def process_tiles(img_tensor, model, tile=512, tile_pad=10, scale=4):
    batch, channel, height, width = img_tensor.shape
    if tile == 0 or (height <= tile and width <= tile):
        with torch.no_grad():
            return model(img_tensor)
    out_tensor = torch.zeros((batch, channel, height * scale, width * scale), dtype=img_tensor.dtype, device=img_tensor.device)
    tiles_x = math.ceil(width / tile)
    tiles_y = math.ceil(height / tile)
    for y in range(tiles_y):
        for x in range(tiles_x):
            in_x = x * tile
            in_y = y * tile
            in_x_end = min(in_x + tile, width)
            in_y_end = min(in_y + tile, height)
            in_x_pad = max(in_x - tile_pad, 0)
            in_x_end_pad = min(in_x_end + tile_pad, width)
            in_y_pad = max(in_y - tile_pad, 0)
            in_y_end_pad = min(in_y_end + tile_pad, height)
            tile_in = img_tensor[:, :, in_y_pad:in_y_end_pad, in_x_pad:in_x_end_pad]
            with torch.no_grad():
                tile_out = model(tile_in)
            out_x = in_x * scale
            out_x_end = in_x_end * scale
            out_y = in_y * scale
            out_y_end = in_y_end * scale
            t_out_x = (in_x - in_x_pad) * scale
            t_out_x_end = t_out_x + (in_x_end - in_x) * scale
            t_out_y = (in_y - in_y_pad) * scale
            t_out_y_end = t_out_y + (in_y_end - in_y) * scale
            out_tensor[:, :, out_y:out_y_end, out_x:out_x_end] = tile_out[:, :, t_out_y:t_out_y_end, t_out_x:t_out_x_end]
    return out_tensor

def upscale_image(pil_img, model, device, tile=512, fp16=True):
    img = pil_img.convert("RGB")
    np_img = np.array(img, dtype=np.float32) / 255.0
    tensor = torch.from_numpy(np_img).permute(2, 0, 1).unsqueeze(0).to(device)
    if fp16 and device.type == "cuda":
        tensor = tensor.half()
    out_tensor = process_tiles(tensor, model, tile=tile, tile_pad=10, scale=4)
    out_tensor = out_tensor.squeeze(0).float().clamp(0.0, 1.0)
    out_np = (out_tensor.permute(1, 2, 0).cpu().numpy() * 255.0).round().astype(np.uint8)
    return Image.fromarray(out_np)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("-i", "--input", type=str, default="inputs")
    parser.add_argument("-o", "--output", type=str, default="outputs")
    parser.add_argument("-m", "--model_path", type=str, default="weights/RealESRGAN_x4plus.pth")
    parser.add_argument("-t", "--tile", type=int, default=512)
    parser.add_argument("-q", "--quality", type=int, default=96)
    parser.add_argument("--fp16", action="store_true", default=True)
    args = parser.parse_args()
    os.makedirs(args.output, exist_ok=True)
    exts = ("*.png", "*.jpg", "*.jpeg", "*.webp", "*.bmp")
    files = []
    for ext in exts:
        files.extend(glob.glob(os.path.join(args.input, ext)))
        files.extend(glob.glob(os.path.join(args.input, ext.upper())))
    files = sorted(list(set(files)))
    if not files:
        print(f"Tidak ada file gambar di folder '{args.input}'.")
        return
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device} | Memuat model {args.model_path}...")
    model = load_model(args.model_path, device, fp16=args.fp16)
    print(f"Memproses {len(files)} gambar...")
    for i, path in enumerate(files, 1):
        name = os.path.splitext(os.path.basename(path))[0]
        out_path = os.path.join(args.output, f"{name}_upscaled.jpg")
        print(f"[{i}/{len(files)}] {os.path.basename(path)} -> ", end="", flush=True)
        try:
            with Image.open(path) as src_img:
                out_img = upscale_image(src_img, model, device, tile=args.tile, fp16=args.fp16)
                out_img.save(out_path, format="JPEG", quality=args.quality, subsampling=0, optimize=True)
                w, h = out_img.size
                print(f"SELESAI ({w}x{h} px)")
        except Exception as e:
            print(f"GAGAL: {e}")

if __name__ == "__main__":
    main()


In [ ]:
# 4. Upload gambar bahan ke folder inputs/
import os
from google.colab import files

print("Klik 'Choose Files' di bawah ini untuk memilih gambar bahan:")
uploaded = files.upload()
for fn in uploaded.keys():
    os.rename(fn, os.path.join('inputs', fn))
print(f"Berhasil mengunggah {len(uploaded)} gambar ke folder inputs/.")

In [ ]:
# 5. Jalankan Batch Upscaling (4x, Tiling 512, High-Quality JPG)
!python upscale_batch.py --input inputs --output outputs --tile 512 --quality 96

In [ ]:
# 6. Kompres folder outputs & Download hasil
import shutil
from google.colab import files

print("Mengompres hasil ke zip...")
shutil.make_archive("hasil_upscale", "zip", "outputs")
print("Mengunduh hasil_upscale.zip...")
files.download("hasil_upscale.zip")